## Mediating Ontology Pipeline

This notebook implements and evaluates the mediating ontology alignment strategy, extending the `LogMapBIO` workflow with LLM-based semantic validation.

The notebook is organized into five sections.

#### Additional experiments

**Select mediating ontology** uses `MediatingOntologySelector` to load LogMap anchors from `output/logmap/{dataset}/{subset}/logmap_anchors.tsv` and rank the top-7 candidate mediating ontologies per subset.

**Perform LogMapBIO algorithm to produce composed mappings** attempts to replicate the mediating ontology composition pipeline (Algorithm 1 from the thesis) using `LogMapBioRunner`. For each subset and each ranked mediator, it runs the full MC pipeline — computing `M1`, `M2`, composed mappings `MC`, direct LogMap mappings, and the difference `MC minus direct` — writing results to `output/logmapbio/{dataset}_{subset}_{mediator}/`. The final cell merges `MC_minus_direct` mappings across all mediators into a single deduplicated `merged_MC_minus_direct.tsv`.

**Compare mediator mappings against LogMap** evaluates the replicated pipeline outputs (`direct`, `MC_composed`, `MC_minus_direct`) for each mediator against ground truth using `OntologyAlignmentEvaluator`, saving per-system Precision/Recall/F1 metrics.

#### Main pipeline

**Compare true MATCHERBIO mediator mappings against LogMap** evaluates the reference `.jar`-produced `LogMapBIO` composed mappings (`anatomy-all-composed-mappings-labeled.txt`) directly against ground truth, serving as the baseline for the true algorithm output.

**Validate mediator mappings using LLM** applies `LLMValidator` backed by `GeminiApiServer` to the reference `LogMapBIO` composed mappings, running zero-shot `direct_entity` prompting per candidate pair and evaluating the LLM-filtered results against ground truth via `OntologyAlignmentEvaluator`.

\
**Note:** The pipeline automatically stores all intermediate and generated results in the `output/` directory, while finalized and manually curated results are saved in the `final_results/` directory.

### Select mediating ontology

In [ ]:
from modules.mediating_selector import MediatingOntologySelector
from utils.utils import format_subsets_ontologies_paths
from pathlib import Path
import os

ALL_DATASET_NAMES = {
    "anatomy": ["human-mouse"],
    "bioml-2024": ["snomed-fma.body", "snomed-ncit.neoplas", "snomed-ncit.pharm", "ncit-doid", "omim-ordo"],
    "largebio": ["fma-snomed", "snomed-nci", "fma-nci"],
    "largebio_small": ["fma-nci", "snomed-nci", "fma-nci"],
}

anchors_path = "output/logmap_anchors.tsv"

for dataset_name, subfolders in ALL_DATASET_NAMES.items():
    for subfolder in subfolders:
        o1_path, o2_path = format_subsets_ontologies_paths(dataset_name, subfolder)
        anchors_path = f"output/logmap/{dataset_name}/{subfolder}/logmap_anchors.tsv"

        lexical_mappings = MediatingOntologySelector.load_logmap_anchors(anchors_path)

        selector = MediatingOntologySelector(
            o1_path=o1_path,
            o2_path=o2_path,
            lexical_mappings=lexical_mappings,
            json_out=f"output/mediators/mediating_ontology_ranking_{dataset_name}_{subfolder}.json"
        )

        mediators = selector.select_top_mediators(top_k=7, download=True)
        print(mediators)

INFO: Loaded 1031 lexical mappings from output/logmap/anatomy/human-mouse/logmap_anchors.tsv
INFO: Classifying ontology with Pellet...
* Owlready2 * Saving world <owlready2.namespace.World object at 0x10befb610> to /var/folders/8b/bqvq8bfs4sv9n07lk0_xl8hr0000gn/T/tmpf6s70ys4...
* Owlready2 * Running Pellet...
    java -Xmx2000M -cp /Users/shuma/Desktop/dyplom/venv/lib/python3.9/site-packages/owlready2/pellet/httpclient-4.2.3.jar:/Users/shuma/Desktop/dyplom/venv/lib/python3.9/site-packages/owlready2/pellet/aterm-java-1.6.jar:/Users/shuma/Desktop/dyplom/venv/lib/python3.9/site-packages/owlready2/pellet/xercesImpl-2.10.0.jar:/Users/shuma/Desktop/dyplom/venv/lib/python3.9/site-packages/owlready2/pellet/slf4j-api-1.6.4.jar:/Users/shuma/Desktop/dyplom/venv/lib/python3.9/site-packages/owlready2/pellet/jena-tdb-0.10.0.jar:/Users/shuma/Desktop/dyplom/venv/lib/python3.9/site-packages/owlready2/pellet/jena-iri-0.9.5.jar:/Users/shuma/Desktop/dyplom/venv/lib/python3.9/site-packages/owlready2/pellet

Ontology successfully classified.


INFO: Classifying ontology with Pellet...
* Owlready2 * Saving world <owlready2.namespace.World object at 0x10befb610> to /var/folders/8b/bqvq8bfs4sv9n07lk0_xl8hr0000gn/T/tmpotnjf6b2...
* Owlready2 * Running Pellet...
    java -Xmx2000M -cp /Users/shuma/Desktop/dyplom/venv/lib/python3.9/site-packages/owlready2/pellet/httpclient-4.2.3.jar:/Users/shuma/Desktop/dyplom/venv/lib/python3.9/site-packages/owlready2/pellet/aterm-java-1.6.jar:/Users/shuma/Desktop/dyplom/venv/lib/python3.9/site-packages/owlready2/pellet/xercesImpl-2.10.0.jar:/Users/shuma/Desktop/dyplom/venv/lib/python3.9/site-packages/owlready2/pellet/slf4j-api-1.6.4.jar:/Users/shuma/Desktop/dyplom/venv/lib/python3.9/site-packages/owlready2/pellet/jena-tdb-0.10.0.jar:/Users/shuma/Desktop/dyplom/venv/lib/python3.9/site-packages/owlready2/pellet/jena-iri-0.9.5.jar:/Users/shuma/Desktop/dyplom/venv/lib/python3.9/site-packages/owlready2/pellet/owlapi-distribution-3.4.3-bin.jar:/Users/shuma/Desktop/dyplom/venv/lib/python3.9/site-packag

Ontology successfully classified.


INFO: MediatingOntologySelector initialized
INFO: Starting mediating ontology selection pipeline
INFO: Extracting labels from lexical mappings
INFO: Collected 1867 representative labels
INFO: Querying BioPortal for mediating candidates
Processing labels:  15%|█▍        | 271/1867 [11:37<1:08:25,  2.57s/it]
INFO: Collected stats for 128 candidate ontologies
INFO: Ranking mediating ontologies
INFO: Saving ranking to output/mediators/mediating_ontology_ranking_anatomy_human-mouse.json
INFO: Using cached ontology: NCIT
INFO: Using cached ontology: UBERON
INFO: Using cached ontology: NIFSTD
INFO: Downloading mediating ontology: SNOMEDCT
INFO: Using cached ontology: UPHENO
INFO: Using cached ontology: FMA
INFO: Downloading mediating ontology: IOBC
INFO: Mediating ontology selection completed


[{'acronym': 'NCIT', 'positive_hits': 132, 'avg_synonyms': 2.840909090909091, 'local_path': 'output/mediators/NCIT.owl'}, {'acronym': 'UBERON', 'positive_hits': 116, 'avg_synonyms': 2.560344827586207, 'local_path': 'output/mediators/UBERON.owl'}, {'acronym': 'NIFSTD', 'positive_hits': 106, 'avg_synonyms': 4.528301886792453, 'local_path': 'output/mediators/NIFSTD.owl'}, {'acronym': 'SNOMEDCT', 'positive_hits': 94, 'avg_synonyms': 1.627659574468085}, {'acronym': 'UPHENO', 'positive_hits': 89, 'avg_synonyms': 2.2808988764044944, 'local_path': 'output/mediators/UPHENO.owl'}, {'acronym': 'FMA', 'positive_hits': 79, 'avg_synonyms': 1.4430379746835442, 'local_path': 'output/mediators/FMA.owl'}, {'acronym': 'IOBC', 'positive_hits': 66, 'avg_synonyms': 4.5}]


### Perform LogMapBio algorithm to produce compose mappings

In [ ]:
from modules.logmap_bio_compose_runner import LogMapBioRunner
from utils.utils import format_subsets_ontologies_paths
import json

ALL_DATASET_NAMES = {
    "anatomy": ["human-mouse"],
    "bioml-2024": ["snomed-fma.body", "snomed-ncit.neoplas", "snomed-ncit.pharm", "ncit-doid", "omim-ordo"],
    "largebio": ["fma-snomed", "snomed-nci", "fma-nci"],
    "largebio_small": ["fma-nci", "snomed-nci", "fma-nci"],
}

for dataset_name, subfolders in ALL_DATASET_NAMES.items():
    for subfolder in subfolders:
        o1_path, o2_path = format_subsets_ontologies_paths(dataset_name, subfolder)
        mos_path = f"output/mediators/mediating_ontology_ranking_{dataset_name}_{subfolder}.json"
        with open(mos_path) as f:
            data = json.load(f)

        owl_files = [entry["acronym"] + ".owl" for entry in data]

        for mo in owl_files:
            runner = LogMapBioRunner(
                o1_path=o1_path,
                o2_path=o2_path,
                mediating_ontology_path=f"output/mediators/{mo}",
                work_dir="output/logmapbio/" + dataset_name + "_" + subfolder + "_" + mo.replace(".owl", "")
            )

            # Run composition only (step 3)
            # results_compose = runner.run_composition_only()

            # Run full pipeline (MC - Direct)
            results_full = runner.run()

            print("=== Full MC Pipeline ===")
            print(f"M1 output: {results_full['M1_path']}")
            print(f"M2 output: {results_full['M2_path']}")
            print(f"MC composed mappings: {results_full['MC_file']}")
            print(f"Direct LogMap mappings: {results_full['Direct_file']}")
            print(f"MC minus direct mappings: {results_full['MC_minus_direct_file']}")
            print(f"MC size: {results_full['MC_size']}")
            print(f"Direct size: {results_full['Direct_size']}")
            print(f"MC - Direct size: {results_full['MC_minus_size']}")

INFO: LogMapBioRunner initialized
INFO: Starting mediating alignment (composition-only mode)
INFO: Running LogMap MATCHER locally: /Library/Java/JavaVirtualMachines/temurin-8.jdk/Contents/Home/bin/java -Xmx12g -jar /Users/shuma/Desktop/dyplom/modules/logmap/logmap-matcher-4.0.jar MATCHER file:data/anatomy/human-mouse/mouse.owl file:output/mediators/UPHENO.owl /Users/shuma/Desktop/dyplom/output/logmapbio/anatomy_human-mouse_UPHENO/step1_o1_mo/ true
Error reading LogMap parameters. File 'parameters.txt' is not available. Using default parameters.



Num unsat classes after integration: 0


INFO: Pairwise alignment completed: step1_o1_mo
INFO: Running LogMap MATCHER locally: /Library/Java/JavaVirtualMachines/temurin-8.jdk/Contents/Home/bin/java -Xmx12g -jar /Users/shuma/Desktop/dyplom/modules/logmap/logmap-matcher-4.0.jar MATCHER file:output/mediators/UPHENO.owl file:data/anatomy/human-mouse/human.owl /Users/shuma/Desktop/dyplom/output/logmapbio/anatomy_human-mouse_UPHENO/step2_mo_o2/ true
Error reading LogMap parameters. File 'parameters.txt' is not available. Using default parameters.



Num unsat classes after integration: 0


INFO: Pairwise alignment completed: step2_mo_o2
INFO: Composing mappings via mediating ontology
INFO: Composed 1244 mappings
INFO: Running direct LogMap(O1, O2)
INFO: Running LogMap MATCHER locally: /Library/Java/JavaVirtualMachines/temurin-8.jdk/Contents/Home/bin/java -Xmx12g -jar /Users/shuma/Desktop/dyplom/modules/logmap/logmap-matcher-4.0.jar MATCHER file:data/anatomy/human-mouse/mouse.owl file:data/anatomy/human-mouse/human.owl /Users/shuma/Desktop/dyplom/output/logmapbio/anatomy_human-mouse_UPHENO/direct_o1_o2/ true
Error reading LogMap parameters. File 'parameters.txt' is not available. Using default parameters.



Num unsat classes after integration: 238


INFO: MC size: 1244
INFO: Direct size: 1405
INFO: MC - Direct size: 195


=== Full MC Pipeline ===
M1 output: /Users/shuma/Desktop/dyplom/output/logmapbio/anatomy_human-mouse_UPHENO/step1_o1_mo/logmap_mappings.rdf
M2 output: /Users/shuma/Desktop/dyplom/output/logmapbio/anatomy_human-mouse_UPHENO/step2_mo_o2/logmap_mappings.rdf
MC composed mappings: output/logmapbio/anatomy_human-mouse_UPHENO/MC_composed.txt
Direct LogMap mappings: output/logmapbio/anatomy_human-mouse_UPHENO/direct_mappings.txt
MC minus direct mappings: output/logmapbio/anatomy_human-mouse_UPHENO/MC_minus_direct.txt
MC size: 1244
Direct size: 1405
MC - Direct size: 195


Now join produced `MC - Direct size` mappings.

In [2]:
# for anatomy example, modify for different datasets

import csv
from pathlib import Path

BASE_DIR = Path("output/results_mediator")
TIMESTAMP_DIR = sorted(BASE_DIR.iterdir())[-1]  # latest run
ANATOMY_DIR = TIMESTAMP_DIR / "anatomy"

ontologies = ["FMA", "NCIT", "NIFSTD", "UBERON", "UPHENO"]

all_mappings = set()

for ont in ontologies:
    minus_dir = ANATOMY_DIR / f"{ont}_MC_minus_direct"
    csv_file = minus_dir / "detailed_results.csv"
    if not csv_file.exists():
        print(f"Skipping missing file: {csv_file}")
        continue

    with open(csv_file, newline='', encoding='utf-8') as f:
        reader = csv.reader(f)
        header = next(reader)  # skip header
        for row in reader:
            src = row[0].strip()
            tgt = row[1].strip()
            all_mappings.add((src, tgt))  # deduplicated automatically

# Save merged file
merged_file = ANATOMY_DIR / "merged_MC_minus_direct.tsv"
with open(merged_file, "w", encoding="utf-8") as f:
    for src, tgt in sorted(all_mappings):
        f.write(f"{src}\t{tgt}\n")

print(f"Merged {len(all_mappings)} unique minus-direct mappings saved to {merged_file}")

Merged 410 unique minus-direct mappings saved to output/results_mediator/2026-03-03_13-48/anatomy/merged_MC_minus_direct.tsv


### Compare mediator mappings aganst logmap

In [ ]:
from modules.evaluator import OntologyAlignmentEvaluator
import pandas as pd
from datetime import datetime
from pathlib import Path
import os

# ============================================================
# CONFIG
# ============================================================

DATASET_NAME = "anatomy"
GT_PATH = Path("data/anatomy/human-mouse/refs_equiv/full.tsv")

OUTPUT_DIR = Path("output/results_mediator")
BASE_MEDIATOR_DIR = Path("output/logmapbio")

MEDIATORS = ["FMA", "NCIT", "NIFSTD", "UBERON", "UPHENO"]

PRED_COL = "Prediction"

# ============================================================
# LOAD LOGMAP TXT FILE
# ============================================================

def load_logmap_file(file_path: Path) -> pd.DataFrame:
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    records = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("|")
            if len(parts) < 2:
                continue

            src, tgt = parts[0], parts[1]
            score = float(parts[3]) if len(parts) > 3 else 1.0
            records.append({
                "Source": src,
                "Target": tgt,
                "LogMapScore": score,
                "Prediction": score > 0
            })

    return pd.DataFrame(records)

# ============================================================
# INITIALIZE EVALUATOR
# ============================================================

print("\nInitializing OntologyAlignmentEvaluator...")
evaluator = OntologyAlignmentEvaluator(str(GT_PATH))

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
results_path = OUTPUT_DIR / timestamp / DATASET_NAME
results_path.mkdir(parents=True, exist_ok=True)

summary_rows = []

# ============================================================
# EVALUATE ALL MEDIATORS
# ============================================================

for mediator in MEDIATORS:
    print(f"\n{'='*30}\nMEDIATOR: {mediator}\n{'='*30}")

    mediator_dir = BASE_MEDIATOR_DIR / f"{DATASET_NAME}_human-mouse_{mediator}"
    systems = {
        f"{mediator}_LogMap_direct": mediator_dir / "direct_mappings.txt",
        f"{mediator}_MC_composed": mediator_dir / "MC_composed.txt",
        f"{mediator}_MC_minus_direct": mediator_dir / "MC_minus_direct.txt",
    }

    for system_name, path in systems.items():
        if not path.exists():
            print(f"Skipping missing file: {path}")
            continue

        print(f"\nEvaluating {system_name}...")
        try:
            df = load_logmap_file(path)

            # Use the new mediator-specific evaluation method
            metrics = evaluator.evaluate_mediator(
                df=df,
                dataset_name=DATASET_NAME,
                experiment_type=system_name,
                system_name=system_name,
                results_dir=OUTPUT_DIR,
            )[system_name]

            summary_rows.append({
                "System": system_name,
                **{k: v for k, v in metrics.items() if k != "ConfusionMatrix"}
            })

        except Exception as e:
            print(f"Error evaluating {system_name}: {e}")

# ============================================================
# SAVE FINAL METRICS
# ============================================================

metrics_df = pd.DataFrame(summary_rows)

columns_order = [
    "System",
    "Precision",
    "Recall",
    "F1",
    "TP",
    "TN",
    "FP",
    "FN",
    "Sensitivity",
    "Specificity",
    "YoudenIndex",
]

# Ensure all expected columns exist
for col in columns_order:
    if col not in metrics_df.columns:
        metrics_df[col] = pd.NA

metrics_df = metrics_df[columns_order]

metrics_csv = results_path / "metrics_comparison.csv"
metrics_df.to_csv(metrics_csv, index=False)

print("\n=== FINAL METRICS ===")
print(metrics_df.to_string(index=False))

if not metrics_df.empty:
    best_method = metrics_df.sort_values(by="F1", ascending=False).iloc[0]["System"]
    print(f"\nBest method based on F1 → {best_method}")
else:
    print("\nNo metrics were computed. Check input files.")


Initializing OntologyAlignmentEvaluator...

MEDIATOR: FMA

Evaluating FMA_LogMap_direct...

Evaluating FMA_MC_composed...

Evaluating FMA_MC_minus_direct...

MEDIATOR: NCIT

Evaluating NCIT_LogMap_direct...

Evaluating NCIT_MC_composed...

Evaluating NCIT_MC_minus_direct...

MEDIATOR: NIFSTD

Evaluating NIFSTD_LogMap_direct...

Evaluating NIFSTD_MC_composed...

Evaluating NIFSTD_MC_minus_direct...
[WARN] Failed to compute Label: Cannot set a DataFrame without columns to the column Label
[WARN] Failed to compute LogMapPred: 'LogMapScore'


/Users/shuma/Desktop/dyplom/venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(



MEDIATOR: UBERON

Evaluating UBERON_LogMap_direct...

Evaluating UBERON_MC_composed...

Evaluating UBERON_MC_minus_direct...

MEDIATOR: UPHENO

Evaluating UPHENO_LogMap_direct...

Evaluating UPHENO_MC_composed...

Evaluating UPHENO_MC_minus_direct...

=== FINAL METRICS ===
                System  Precision  Recall       F1   TP  TN  FP  FN  Sensitivity  Specificity  YoudenIndex
     FMA_LogMap_direct   0.915302     1.0 0.955779 1286   0 119   0          1.0          0.0          0.0
       FMA_MC_composed   0.910927     1.0 0.953388  992   0  97   0          1.0          0.0          0.0
   FMA_MC_minus_direct   0.540741     1.0 0.701923   73   0  62   0          1.0          0.0          0.0
    NCIT_LogMap_direct   0.915302     1.0 0.955779 1286   0 119   0          1.0          0.0          0.0
      NCIT_MC_composed   0.917445     1.0 0.956946 1178   0 106   0          1.0          0.0          0.0
  NCIT_MC_minus_direct   0.147059     1.0 0.256410    5   0  29   0          1.0   

### Compare true MATCHERBIO mediator mappings aganst logmap

In [ ]:
import pandas as pd
from datetime import datetime
from pathlib import Path
from modules.evaluator import OntologyAlignmentEvaluator

DATASET_NAME = "anatomy"
GT_PATH = Path("data/anatomy/human-mouse/refs_equiv/full.tsv")
OUTPUT_DIR = Path("output/results_mediator/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def load_labeled_mappings(file_path: Path, divisor: str = ",") -> pd.DataFrame:
    """
    Load mappings from a .txt file with columns:
    Source, Target, [Score], Prediction (True/False)
    Keep only Source, Target, Prediction.
    """
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    records = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split(divisor)
            if len(parts) < 2:
                continue

            src, tgt = parts[0], parts[1]
            # Prediction is last column
            prediction = parts[-1].strip().lower() == "true" if len(parts) > 2 else True

            records.append({
                "Source": src,
                "Target": tgt,
                "Prediction": prediction
            })

    return pd.DataFrame(records)


print("\nInitializing OntologyAlignmentEvaluator...")
evaluator = OntologyAlignmentEvaluator(str(GT_PATH))

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
results_path = OUTPUT_DIR / timestamp / DATASET_NAME
results_path.mkdir(parents=True, exist_ok=True)

summary_rows = []

# ============================================================
# EVALUATE FILE
# ============================================================

mapping_file = Path(
    "final_results/mediate_ontologies/original_logmapbio/composed_mappings/logmapbio_mappings/anatomy-all-composed-mappings-labeled.txt"
)

print(f"\nEvaluating file: {mapping_file}")
df = load_labeled_mappings(mapping_file)
print(df.head())

# Keep only necessary columns
df = df[["Source", "Target", "Prediction"]]

# Evaluate
metrics = evaluator.evaluate_labeled_mappings(
    df,
    dataset_name=DATASET_NAME,
    system_name="anatomy_all_composed_mappings",
    results_dir=OUTPUT_DIR
)["anatomy_all_composed_mappings"]

summary_rows.append({
    "System": "anatomy_all_composed_mappings",
    **{k: v for k, v in metrics.items() if k != "ConfusionMatrix"}
})

# ============================================================
# SAVE FINAL METRICS
# ============================================================

metrics_df = pd.DataFrame(summary_rows)

columns_order = [
    "System",
    "Precision",
    "Recall",
    "F1",
    "TP",
    "TN",
    "FP",
    "FN",
    "Sensitivity",
    "Specificity",
    "YoudenIndex",
]

# Ensure all expected columns exist
for col in columns_order:
    if col not in metrics_df.columns:
        metrics_df[col] = pd.NA

metrics_df = metrics_df[columns_order]

metrics_csv = results_path / "metrics_comparison.csv"
metrics_df.to_csv(metrics_csv, index=False)

print("\n=== FINAL METRICS ===")
print(metrics_df.to_string(index=False))

if not metrics_df.empty:
    best_method = metrics_df.sort_values(by="F1", ascending=False).iloc[0]["System"]
    print(f"\nBest method based on F1 → {best_method}")
else:
    print("\nNo metrics were computed. Check input files.")


Initializing OntologyAlignmentEvaluator...

Evaluating file: final_results/mediate_ontologies/original_logmapbio/composed_mappings/logmapbio_mappings/anatomy-all-composed-mappings-labeled.txt
                        Source                       Target  Prediction
0  http://mouse.owl#MA_0000322  http://human.owl#NCI_C32461       False
1  http://mouse.owl#MA_0000381  http://human.owl#NCI_C12722        True
2  http://mouse.owl#MA_0001513  http://human.owl#NCI_C32577       False
3  http://mouse.owl#MA_0000717  http://human.owl#NCI_C13053        True
4  http://mouse.owl#MA_0001693  http://human.owl#NCI_C32211        True

=== FINAL METRICS ===
                       System  Precision   Recall       F1  TP  TN  FP  FN  Sensitivity  Specificity  YoudenIndex
anatomy_all_composed_mappings   0.572289 0.650685 0.608974  95 177  71  51     0.650685      0.71371     0.364395

Best method based on F1 → anatomy_all_composed_mappings


### Validate mediator mappings using LLM

In [ ]:
import os
import pandas as pd
import tqdm

from utils.rag import expand_setups
from utils.utils import format_subsets_ontologies_paths
from utils.onto_object import OntologyEntryAttr, OntologyAccess

from modules.llm_validator import LLMValidator
from modules.evaluator import OntologyAlignmentEvaluator
from utils.llm_server.gemini import GeminiApiServer


# -------------------------
# DATASETS
# -------------------------

ALL_DATASET_NAMES = {
    "anatomy": ["human-mouse"],
}


# -------------------------
# EXPERIMENT SETUPS
# -------------------------

BASE_SETUP = {
    "subset": "human-mouse",
    # "model": "meta-llama/llama-3.1-70b-instruct",
    # "model": "qwen/qwen3-vl-8b-instruct",
    # "model": "google/gemini-2.5-pro",
    "model": "gemini-2.5-pro",
    "max_workers": 1,
    "suffix": "_reduced",
    "prompt_functions": ["direct_entity"],
}

SETUPS = [
    {"experiment_type": "zero_shot", "k": 0},
]

RUNS = [
    "2026-03-03_13-48"
]

input_file = "final_results/mediate_ontologies/original_logmapbio/composed_mappings/logmapbio_mappings/anatomy-all-composed-mappings.txt"
DIVISOR = "," # "\t"

SETUPS = expand_setups(SETUPS, BASE_SETUP)

print("Expanded setups:")
for s in SETUPS:
    print(s)


# -------------------------
# RUN EXPERIMENTS
# -------------------------

for dataset_name, subfolders in ALL_DATASET_NAMES.items():

    for subfolder in subfolders:

        print(f"\nRunning dataset {dataset_name}/{subfolder}")

        o1_path, o2_path = format_subsets_ontologies_paths(dataset_name, subfolder)

        gt_path = f"data/{dataset_name}/{subfolder}/refs_equiv/full.tsv"

        # mediator file instead of oracle queries
        # input_file = f"output/results_mediator/{RUNS[0]}/{dataset_name}/merged_MC_minus_direct.tsv"

        onto_src = OntologyAccess(o1_path, annotate_on_init=True)
        onto_tgt = OntologyAccess(o2_path, annotate_on_init=True)

        validator = LLMValidator(llm_server=GeminiApiServer())

        with open(input_file, "r") as f:
            lines = f.readlines()

        print("Mappings loaded:", len(lines))


        for setup in SETUPS:

            experiment_type = setup["experiment_type"]
            k = setup["k"]
            model = setup["model"]

            prompt_type = setup["prompt_functions"]

            print(
                f"\nRunning setup | dataset={dataset_name}"
                f" | type={experiment_type} | k={k} | model={model}"
            )

            results = []

            # -------------------------
            # VALIDATION LOOP
            # -------------------------

            for line in tqdm.tqdm(
                lines,
                desc=f"{experiment_type}_k{k}"
            ):

                parts = line.strip().split(DIVISOR)

                if len(parts) < 2:
                    continue

                tgt_uri = parts[0]
                src_uri = parts[1]

                # logmap_score = float(parts[3]) if len(parts) > 3 else None
                logmap_score = None

                src_entity = OntologyEntryAttr(class_uri=tgt_uri, onto=onto_src)
                tgt_entity = OntologyEntryAttr(class_uri=src_uri, onto=onto_tgt)


                if experiment_type == "zero_shot":

                    res = validator.validate(
                        src_entity,
                        tgt_entity,
                        prompt_type=prompt_type,
                        model=model,
                    )

                else:
                    raise ValueError(f"Unknown experiment type {experiment_type}")

                results.append({
                    "Source": src_uri,
                    "Target": tgt_uri,

                    "LLMDecision": res["is_match"],
                    "LLMConfidence": res["confidence"],
                    "LLMTotalTokens": res["tokens_used"],

                    "LogMapScore": logmap_score,

                    "ExperimentType": experiment_type,
                    "K": k,
                    "Model": model,
                    "Prompt": prompt_type
                })


            # -------------------------
            # SAVE RESULTS
            # -------------------------

            df = pd.DataFrame(results)

            output_dir = f"output/results/{dataset_name}/{experiment_type}_mediator"
            os.makedirs(output_dir, exist_ok=True)

            df.to_csv(f"{output_dir}/mediator_predictions.csv", index=False)


            # -------------------------
            # EVALUATION
            # -------------------------

            evaluator = OntologyAlignmentEvaluator(gt_path)

            report = evaluator.evaluate(
                df=df,
                dataset_name=dataset_name,
                experiment_type=f"{experiment_type}_mediator",
                prompts_used=prompt_type,
                second_system_pred_col="LLMDecision",
                results_dir="output/results"
            )

            print("\n=== Evaluation Report ===")
            print(report)